In [ ]:
# Written by tools/build_notebooks.py -- do not edit. The bootstrap cell below
# compares this against the repo it clones and tells you if these cells are old.
CELLS_SRC = "kaggle_00_build_dataset.py"
CELLS_SHA = "1e299a79091533d1"

# 00 — Build the dataset

**Accelerator: None (CPU).** This notebook spends **no GPU quota**. Do not turn a GPU on —
you would burn your 30 hours a week on `soundfile`.

**Runtime: about 10–15 minutes.** You run this notebook **once, ever.**

## What it does, in plain words

Libri2Mix ships the *separate* voices (`s1/`, `s2/`), not only the mixed audio. This notebook
copies every one of those voices into **one big flat file per split**, and then writes the
dev and test sets as **recipe CSVs** — short text files that say "take utterance 412, crop it
at sample 8000, add white noise at 12 dB" and can rebuild the exact audio later.

| why | what you get |
|---|---|
| **Disk** | ~4 GB instead of the ~32 GB it would take to save every mixture as a WAV |
| **Speed** | one long sequential read instead of 100,000 tiny file opens |
| **Any N** | mixtures for 1–5 speakers are built on the fly, so nothing is baked in |
| **Frozen tests** | the test set is a few hundred kB of text you can put in git |

## Before you press Run

1. **Settings → Accelerator → None**
2. **Settings → Internet → On** (only needed if the code comes from GitHub)
3. **+ Add Input → Datasets → search `libri2mix-8khz-min`** (by `unconscious`, 9.96 GB)

## After it finishes

**Save Version → Save & Run All (Commit).** The output becomes a dataset you attach to every
later notebook. See `docs/KAGGLE_RUNBOOK.md` step 3.

## Bootstrap (this cell is identical in every notebook)

Three ways to get the code onto the Kaggle machine, tried in order:

1. **GitHub clone** — set `REPO_URL` below and turn *Internet* ON in the notebook
   settings panel (Settings → Internet → On). This is the recommended route.
2. **Repo-as-dataset** — upload this folder as a Kaggle Dataset called
   `speaker-count-separate-v1` and attach it. No internet needed. Use this if your
   account cannot enable internet (phone-verification is required for that).
3. **Already there** — an existing clone is **fast-forwarded to the newest commit**,
   not reused as-is. A Kaggle session outlives many pushes, and silently running code
   from an hour ago is the most expensive kind of confusion: the log looks fine and the
   fix you are testing is not in it. Any local edits inside the clone are discarded.

Whichever route runs, the commit is printed. Every log can then be traced to the exact
code that produced it.

In [ ]:
REPO_URL = "https://github.com/AlAminAshraf01/speaker-count-separate-v1.git"
REPO_DIR = "/kaggle/working/speaker-count-separate-v1"
REPO_AS_DATASET = "/kaggle/input/speaker-count-separate-v1"

import hashlib
import os
import shutil
import subprocess
import sys


def cells_fingerprint(src_dir: str, name: str) -> str:
    """Short hash of one notebook's percent source plus this shared bootstrap.

    ``tools/build_notebooks.py`` stamps this into every generated ``.ipynb``. The copy
    running on Kaggle recomputes it from the freshly-cloned repo, so a notebook whose
    cells were imported before the last push says so in the first ten seconds instead of
    eleven hours later.

    Line endings are normalised first. The same file is CRLF in a Windows working tree
    and LF in a Linux clone, and a fingerprint that disagrees with itself across
    platforms is worse than no fingerprint at all.
    """
    digest = hashlib.sha256()
    for part in (name, "_bootstrap.py"):
        with open(os.path.join(src_dir, part), "rb") as fh:
            digest.update(fh.read().replace(b"\r\n", b"\n"))
        digest.update(b"\0")
    return digest.hexdigest()[:16]


def cells_status(repo_dir: str, src_name: str | None, stamp: str | None) -> str:
    """Compare the stamp baked into these cells with the repo they are about to run.

    Never raises. A check that can take down every notebook is a worse bug than the one
    it detects, so anything unreadable degrades to "cannot verify".
    """
    if not src_name or not stamp:
        return "unstamped -- re-import this notebook to enable the staleness check"
    try:
        current = cells_fingerprint(os.path.join(repo_dir, "notebooks", "src"), src_name)
    except Exception as exc:
        return f"cannot verify ({exc})"
    if current == stamp:
        return f"current ({stamp})"
    return "\n".join([
        f"STALE  cells {stamp} but repo has {current}",
        "",
        "  These notebook cells were imported before the newest push, so the fix you",
        "  are about to test is not in them. scripts/ and src/ just updated themselves;",
        "  notebook cells cannot, because Kaggle owns them.",
        "",
        "  Fix: File -> Import Notebook -> upload notebooks/" + src_name[:-3] + ".ipynb",
        "       again, re-attach the inputs, and re-run.",
    ])


def _git(repo_dir: str, *argv: str) -> subprocess.CompletedProcess:
    return subprocess.run(["git", "-C", repo_dir, *argv],
                          capture_output=True, text=True)


def update_clone(repo_dir: str) -> str:
    """Fast-forward an existing clone to the remote's newest commit.

    Returns a short status for printing; never raises. Losing internet is a reason to
    carry on with the code that is already there, but it is not a reason to be quiet
    about it -- running stale code unknowingly is how a fix gets tested without being
    present.
    """
    if not os.path.isdir(os.path.join(repo_dir, ".git")):
        return "not a git clone, left as it is"
    branch = _git(repo_dir, "rev-parse", "--abbrev-ref", "HEAD").stdout.strip() or "main"
    before = _git(repo_dir, "rev-parse", "--short", "HEAD").stdout.strip()
    fetched = _git(repo_dir, "fetch", "--depth", "1", "origin", branch)
    if fetched.returncode != 0:
        tail = (fetched.stderr or "").strip().splitlines()
        return f"COULD NOT FETCH ({tail[-1] if tail else 'unknown'}) -- code may be stale"
    reset = _git(repo_dir, "reset", "--hard", f"origin/{branch}")
    if reset.returncode != 0:
        tail = (reset.stderr or "").strip().splitlines()
        return f"COULD NOT UPDATE ({tail[-1] if tail else 'unknown'}) -- code may be stale"
    after = _git(repo_dir, "rev-parse", "--short", "HEAD").stdout.strip()
    return "already newest" if before == after else f"updated {before} -> {after}"


def describe_commit(repo_dir: str) -> str:
    """``<short sha> <date> <subject>`` for the checked-out commit, or a plain note."""
    out = _git(repo_dir, "log", "-1", "--format=%h %cs %s").stdout.strip()
    return out or "no git metadata"


def bootstrap(repo_url: str = REPO_URL, repo_dir: str = REPO_DIR) -> str:
    """Put the repo at `repo_dir`, put its `src/` on sys.path, and chdir into it."""
    if not os.path.isdir(os.path.join(repo_dir, "src")):
        if os.path.isdir(os.path.join(REPO_AS_DATASET, "src")):
            shutil.copytree(REPO_AS_DATASET, repo_dir, dirs_exist_ok=True)
            print(f"copied repo from the attached dataset {REPO_AS_DATASET}")
        else:
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_dir], check=True)
            print(f"cloned {repo_url}")
    else:
        print(f"existing clone: {update_clone(repo_dir)}")
    src = os.path.join(repo_dir, "src")
    if src not in sys.path:
        sys.path.insert(0, src)
    os.chdir(repo_dir)
    return repo_dir


REPO = bootstrap()

import countsep  # noqa: E402

print("countsep", countsep.__version__, "at", REPO)
print("code ", describe_commit(REPO))
# CELLS_SRC / CELLS_SHA are set by the stamp cell that tools/build_notebooks.py puts at
# the top of every generated notebook. globals().get keeps this working in a notebook
# assembled by hand, where that cell may not exist.
CELLS = cells_status(REPO, globals().get("CELLS_SRC"), globals().get("CELLS_SHA"))
print("cells", CELLS if "\n" not in CELLS else "")
if "\n" in CELLS:
    print(CELLS)
print("python", sys.version.split()[0])

import torch  # noqa: E402

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| devices", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  [{i}] {p.name}  {p.total_memory / 1e9:.1f} GB")

In [ ]:
import shlex
import time


def run(cmd: str, check: bool = True) -> int:
    """Run a shell command, streaming its output into the notebook."""
    print("$", cmd, flush=True)
    t0 = time.time()
    proc = subprocess.Popen(shlex.split(cmd), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
    print(f"\n[exit {code} in {time.time() - t0:.1f}s]", flush=True)
    if check and code != 0:
        raise SystemExit(f"command failed with exit code {code}")
    return code

## Check before you spend

Thirty seconds, writes nothing. It also tells you if these notebook cells are older than the
code they just downloaded — which is the most expensive kind of confusion, because the log
looks completely normal.

In [ ]:
import sys
sys.path.insert(0, os.path.join(REPO, "scripts"))

run(f"python scripts/preflight.py --for data"
    f" --cells_src {CELLS_SRC} --cells_sha {CELLS_SHA}")

## Find the Libri2Mix input

`autodetect_libri2mix` searches the usual Kaggle paths by looking at folder *contents*, not
at a fixed path, because Kaggle mounts datasets differently depending on how you added them.
Normally you change nothing here.

In [ ]:
from _common import autodetect_libri2mix

LIBRI2MIX_DIR = autodetect_libri2mix()      # or paste the path yourself
STORE = "/kaggle/working/store"
DATA = "/kaggle/working/data"

print("Libri2Mix :", LIBRI2MIX_DIR)
if LIBRI2MIX_DIR is None:
    raise SystemExit(
        "Not found. In the right-hand panel: '+ Add Input' -> Datasets -> search for\n"
        "'libri2mix-8khz-min' -> Add. Then re-run this cell.")
for split in ("train-100", "dev", "test"):
    path = os.path.join(LIBRI2MIX_DIR, split)
    print(f"  {split:<10} {'ok' if os.path.isdir(path) else 'MISSING'}  {path}")

## Step 1 — pack the voices  (~10 min)

Each utterance is stored as its **loudest 8-second window**. A random 3-second crop of a
LibriSpeech recording often lands in a pause, and a mixture built from three silent crops is
not really a three-speaker mixture.

20 % of the speakers in each split are set aside with `role = babble` and are **never** used
as counting targets. (We do not use babble noise at all in this project — see step 2 — but
the split is kept so the store stays compatible with the sibling repo.)

This step is restartable: a split that is already packed is skipped.

In [ ]:
run(f"python scripts/00_pack_sources.py"
    f" --libri2mix_dir {LIBRI2MIX_DIR}"
    f" --out {STORE}"
    f" --splits train-100 dev test"
    f" --babble_frac 0.2"
    f" --cap_seconds 8.0"
    f" --seed 72")

## Step 2 — freeze the dev and test sets  (~2 min)

These two CSV files **are** the evaluation protocol. Generate them once, commit them, and
never regenerate them — different recipes mean a different test set, and every number you
have already written down becomes incomparable.

### The mixing settings, and why they are not the sibling repo's

| setting | old project | here | why |
|---|---|---|---|
| gain jitter | ±5 dB | **±2.5 dB** | ±5 dB was measured to cost 9 accuracy points |
| SNR | 0–20 dB | **5–20 dB** | below 5 dB the noise drowns the cue we are counting |
| babble noise | 20 % of clips | **removed** | babble *is* 4–8 real talkers, so a "2 speaker" clip with babble has six voices in it and a label that says two |

The script **refuses to run** if you ask for babble. That is deliberate: it is not a tuning
knob, it is a wrong label.

In [ ]:
run(f"python scripts/01_make_frozen_sets.py"
    f" --store {STORE}"
    f" --out {DATA}"
    f" --splits dev test"
    f" --n_list 1 2 3 4 5"
    f" --n_per_class 300"
    f" --noise_kinds white pink brown"
    f" --seed 72")

## Step 3 — copy the recipes where you can find them

Download these two files from the notebook output and commit them to the repo, so the
evaluation protocol lives in git rather than only in a Kaggle output.

In [ ]:
run(f"ls -la {DATA}")
print("\nDownload recipes_dev.csv and recipes_test.csv from the Output tab on the right,")
print("then put them in the repo's data/ folder and commit them.")

## Step 4 — check what you built

The last two lines are the ones that matter. If speaker disjointness is not 0 everywhere,
stop: the same person is in both your training and your test data, and every number you
measure afterwards is meaningless.

In [ ]:
run(f"python scripts/preflight.py --for cpu --store {STORE}")

## Now: Save Version → Save & Run All (Commit)

When it finishes, the whole `/kaggle/working` folder becomes a dataset. In every later
notebook you attach it with **+ Add Input → Notebook Output → this notebook**.

You never run this notebook again.